In [ ]:
import os
import pandas as pd
from fredapi import Fred

In [23]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


In [16]:
pip install fredapi

Note: you may need to restart the kernel to use updated packages.


In [24]:
import requests

In [17]:
os.environ["FRED_API_KEY"] = "656d2723f313fd2fe2e6fceb0b265613"

In [20]:
p0_fred_series = {
    "WEI": "WEI",                  # Weekly Economic Index
    "ICSA": "ICSA",                # Initial Claims
    "T10YIE": "T10YIE",            # 10Y Breakeven Inflation
    "DFII10": "DFII10",            # 10Y Real Yield
    "T10Y3M": "T10Y3M",            # 10Y - 3M yield curve slope
    "SOFR": "SOFR",                # Secured Overnight Financing Rate
    "NFCI": "NFCI",                # Chicago Fed NFCI
    "ANFCI": "ANFCI",              # Adjusted NFCI
    "HY_OAS": "BAMLH0A0HYM2",      # ICE BofA US High Yield OAS
    "VIX": "VIXCLS",               # CBOE VIX
    "EPU": "USEPUINDXD",           # US Daily Economic Policy Uncertainty
}

In [21]:
def get_series_vintage_dates(series_id, api_key,
                             realtime_start=None, realtime_end=None):
    url = "https://api.stlouisfed.org/fred/series/vintagedates"
    params = {
        "series_id": series_id,
        "api_key": api_key,
        "file_type": "json",
        "limit": 10000,
    }
    if realtime_start is not None:
        params["realtime_start"] = realtime_start
    if realtime_end is not None:
        params["realtime_end"] = realtime_end

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    return pd.to_datetime(js["vintage_dates"])

In [32]:
wei = get_series_vintage_dates("WEI", os.environ['FRED_API_KEY'])
wei

DatetimeIndex(['2020-04-16', '2020-04-21', '2020-04-23', '2020-04-28',
               '2020-04-30', '2020-05-05', '2020-05-07', '2020-05-12',
               '2020-05-14', '2020-05-19',
               ...
               '2025-12-31', '2026-01-08', '2026-01-15', '2026-01-22',
               '2026-01-29', '2026-02-05', '2026-02-12', '2026-02-19',
               '2026-02-26', '2026-03-05'],
              dtype='datetime64[ns]', length=397, freq=None)

In [31]:
wei_vdates[0]

Timestamp('2020-04-16 00:00:00')

In [33]:
def get_alfred_series_asof(series_id, vintage_date, api_key,
                           observation_start=None, observation_end=None):
    """
    Pull one ALFRED/FRED series as known on `vintage_date`
    using the observations endpoint + vintage_dates parameter.
    """
    url = "https://api.stlouisfed.org/fred/series/observations"

    params = {
        "series_id": series_id,
        "api_key": api_key,
        "file_type": "json",
        "vintage_dates": vintage_date,   # key fix
    }

    if observation_start is not None:
        params["observation_start"] = observation_start
    if observation_end is not None:
        params["observation_end"] = observation_end

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()

    df = pd.DataFrame(js["observations"])
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    s = df.set_index("date")["value"].sort_index()
    s.name = series_id
    return s

In [34]:
def load_p0_fred_asof(vintage_date, api_key,
                      observation_start=None, observation_end=None,
                      series_map=None):
    series_map = p0_fred_series if series_map is None else series_map

    out = {}
    for name, sid in series_map.items():
        out[name] = get_alfred_series_asof(
            series_id=sid,
            vintage_date=vintage_date,
            api_key=api_key,
            observation_start=observation_start,
            observation_end=observation_end,
        )

    df = pd.concat(out, axis=1).sort_index()
    return df

In [35]:
p0_df = load_p0_fred_asof(
    vintage_date="2026-03-07",
    api_key=os.environ['FRED_API_KEY'],
    observation_start="2000-01-01",
)

print(p0_df.tail())

            WEI  ICSA  T10YIE  DFII10  T10Y3M  SOFR  NFCI  ANFCI  HY_OAS  \
date                                                                       
2026-03-02  NaN   NaN    2.29    1.76    0.33  3.71   NaN    NaN    3.03   
2026-03-03  NaN   NaN    2.29    1.77    0.35  3.70   NaN    NaN    3.08   
2026-03-04  NaN   NaN    2.29    1.80    0.38  3.67   NaN    NaN    2.97   
2026-03-05  NaN   NaN    2.31    1.82    0.43  3.66   NaN    NaN    3.00   
2026-03-06  NaN   NaN    2.35     NaN    0.46   NaN   NaN    NaN     NaN   

              VIX     EPU  
date                       
2026-03-02  21.44  441.57  
2026-03-03  23.57  561.10  
2026-03-04  21.15  223.72  
2026-03-05  23.75  552.76  
2026-03-06    NaN     NaN  


In [36]:
p0_df.head()

,WEI,ICSA,T10YIE,DFII10,T10Y3M,SOFR,NFCI,ANFCI,HY_OAS,VIX,EPU
date,,,,,,,,,,,
2000-01-01,NaN,286000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.04
2000-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,119.36
2000-01-03,NaN,NaN,NaN,NaN,1.10,NaN,NaN,NaN,4.68,24.21,35.73
2000-01-04,NaN,NaN,NaN,NaN,1.06,NaN,NaN,NaN,4.81,27.01,109.31
2000-01-05,NaN,NaN,NaN,NaN,1.18,NaN,NaN,NaN,4.77,26.41,123.22
